# Manejo de Latitud y Longitud en el Conjunto de Datos de Precios de Vivienda de California

## Carga y preparación de los datos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
housing = pd.read_csv("./data/housing.csv") 
train_set, test_set = train_test_split(housing, test_size=0.2,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),
    random_state=42
    )
X_train = train_set.drop("median_house_value", axis=1)
y_train = train_set["median_house_value"].copy() # Almacenar la variable objetivo (etiquetas)

## Comprensión de las coordenadas geográficas en el *conjunto de datos*

Las coordenadas geográficas presentan desafíos únicos para los modelos de Machine Learning:

1. **Relaciones espaciales no lineales**: Los precios de las viviendas no aumentan ni disminuyen de forma monótona con la latitud o la longitud. Un distrito a 37.5° de latitud no es inherentemente "mejor" que uno a 36.5°: la relación es altamente no lineal y dependiente de la ubicación.
2. **Efectos de interacción**: La latitud y la longitud solo tienen sentido *juntas*. El punto (37.7749, -122.4194) es San Francisco, pero ninguna coordenada por sí sola contiene esa información. Los modelos lineales no pueden capturar esta interacción sin una ingeniería de características explícita.
3. **La distancia importa, no la posición absoluta**: Lo que suele importar es la *proximidad* a ubicaciones importantes (costa, centros urbanos, polos de empleo), no las coordenadas brutas en sí mismas.
4. **Problemas de envoltura y escala**: La longitud se envuelve en ±180°, y la distancia real en tierra que representa 1° varía con la latitud.

Para abordar estos desafíos, podemos transformar las coordenadas en características más útiles:

- **Pertenencia a un clúster**: Agrupar distritos en regiones geográficas
- **Características basadas en distancias**: Medir la proximidad a los centros de clúster o a puntos de interés
- **Puntuaciones de similitud**: Usar funciones de kernel para capturar la "cercanía" a ubicaciones importantes

## K-means para agrupamiento geográfico

K-means es un algoritmo de *agrupamiento* (aprendizaje no supervisado) que podríamos utilizar para agrupar distritos en regiones geográficas similares. Para ello, usaremos las coordenadas geográficas de los distritos y los agruparemos en un número fijo de regiones geográficas.

In [ ]:
from sklearn.cluster import KMeans

# Aplicar K-means con k=6 (por ejemplo, para dividir California en 6 regiones)
kmeans = KMeans(n_clusters=6, random_state=42)
X_train['region_cluster'] = kmeans.fit_predict(X_train[['latitude', 'longitude']])
X_train[['latitude', 'longitude', 'region_cluster']].head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(X_train['longitude'], X_train['latitude'], 
            c=X_train['region_cluster'], cmap='viridis',
            s=5, alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 1], kmeans.cluster_centers_[:, 0],
            c='red', s=50, marker='x')
plt.title('Agrupamiento de Distritos de California con K-means')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.show()

Podríamos crear un transformador personalizado para este proceso:

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class RegionClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=6, random_state=42):
        self.n_clusters = n_clusters
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=random_state)
        
    def fit(self, X, y=None):
        """
        Ajusta el modelo KMeans usando las columnas 'latitude' y 'longitude'.
        Se asume que X es un DataFrame de pandas que contiene estas columnas.
        """
        self.kmeans.fit(X[['latitude', 'longitude']])
        return self
        
    def transform(self, X):
        """
        Transforma el DataFrame X añadiendo una columna 'region_cluster' con
        la predicción de clúster para cada registro.
        """
        X_transformed = X.copy()
        X_transformed['region_cluster'] = self.kmeans.predict(X_transformed[['latitude', 'longitude']])
        return X_transformed

In [ ]:
transformer = RegionClusterTransformer(n_clusters=6, random_state=42)transformer.fit(X_train)housing_transformed = transformer.transform(X_train)housing_transformed[['latitude', 'longitude', 'region_cluster']].head()

,latitude,longitude,region_cluster
12655,38.52,-121.46,3
15502,33.09,-117.23,4
2908,35.37,-119.04,5
14053,32.75,-117.13,4
20496,34.28,-118.70,0


## Añadir la distancia a los centroides como característica

Sin embargo, los precios de las viviendas no solo están relacionados con el clúster de distritos al que pertenecen, sino que los distritos más cercanos al centro del clúster probablemente tendrán precios más altos. Por lo tanto, sería útil crear una característica que capture la "proximidad" a los centros de los clústeres.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

class DistanceClusterTransformer(BaseEstimator, TransformerMixin):
    """
    Transformador personalizado para agrupar ubicaciones geográficas usando K-means.
    Espera un DataFrame con al menos dos columnas: 'latitude' y 'longitude'.
    """
    def __init__(self, n_clusters=8, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans_ = None

    def fit(self, X, y=None):
        # Se espera que X sea un dataframe o array con columnas 'latitude' y 'longitude'
        # Convertir a array si X es un dataframe:
        if isinstance(X, pd.DataFrame):
            coords = X[['latitude', 'longitude']].values
        else:
            coords = X[:, :2]  # asumiendo que las dos primeras columnas son lat/lon
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(coords)
        return self

    def transform(self, X):
        # Asegurar que X es un dataframe para un manejo más fácil de las columnas
        X_trans = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=['latitude', 'longitude'])
        coords = X_trans[['latitude', 'longitude']].values
        
        # Predecir la etiqueta del clúster
        cluster_labels = self.kmeans_.predict(coords)
        X_trans['geo_cluster'] = cluster_labels
        
        # Calcular la distancia de cada punto a su centroide asignado
        distances = []
        for i, point in enumerate(coords):
            centroid = self.kmeans_.cluster_centers_[cluster_labels[i]]
            distance = np.linalg.norm(point - centroid)
            distances.append(distance)
        X_trans['distance_to_centroid'] = distances
        
        return X_trans

In [ ]:
geoClusterTransformer = DistanceClusterTransformer(n_clusters=5)
geoClusterTransformer.fit(X_train[['longitude', 'latitude']])
housing_geo_clusters = geoClusterTransformer.transform(X_train[['longitude', 'latitude']])
housing_geo_clusters.head()

## Similitud con rbf_kernel

### rbf_kernel

La función `rbf_kernel` calcula la similitud entre dos conjuntos de datos mediante la *función de base radial* (RBF), devolviendo un valor entre 0 y 1.

Por ejemplo, si asumiéramos que las propiedades de alrededor de 35 años aumentan de valor por alguna particularidad, podríamos medir la distancia de cada una respecto a las de 35 años y asignar un valor de similitud.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernelage_simil_35 = rbf_kernel(X_train[["housing_median_age"]], [[35]], gamma=0.1)

In [ ]:
ages = np.linspace(X_train["housing_median_age"].min(),
                   X_train["housing_median_age"].max(),
                   500).reshape(-1, 1)
gamma1 = 0.1
gamma2 = 0.03
rbf1 = rbf_kernel(ages, [[35]], gamma=gamma1)
rbf2 = rbf_kernel(ages, [[35]], gamma=gamma2)

fig, ax1 = plt.subplots()
ax1.set_xlabel("Antigüedad mediana de la vivienda")
ax1.set_ylabel("Número de distritos")
ax1.hist(X_train["housing_median_age"], bins=50)

ax2 = ax1.twinx()  # crear un eje gemelo que comparte el mismo eje x
color = "blue"
ax2.plot(ages, rbf1, color=color, label="gamma = 0.10")
ax2.plot(ages, rbf2, color=color, label="gamma = 0.03", linestyle="--")
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylabel("Similitud por antigüedad", color=color)
plt.legend(loc="upper left")
plt.show()

Podemos ver cómo el valor de `gamma` permite ajustar la similitud entre los datos, ensanchando o estrechando la función RBF, que es una distribución normal (gaussiana) centrada en el valor de referencia. A la derecha, podemos ver la escala de 0 a 1 de similitud que devolverán los registros cercanos al valor de referencia.

### Medición de la similitud con los centroides mediante rbf_kernel

En lugar de la distancia, podemos usar `rbf_kernel` para medir la proximidad de cada distrito a los centros de los clústeres.

In [ ]:
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # ¡siempre devolver self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    
    def get_feature_names_out(self, names=None):
        return [f"Similitud con clúster {i}" for i in range(self.n_clusters)]

In [ ]:
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)similarities = cluster_simil.fit_transform(X_train[["latitude", "longitude"]],                                           sample_weight=y_train)

In [ ]:
housing_renamed = X_train.rename(columns={
    "latitude": "Latitud", "longitude": "Longitud",
    "population": "Población",
    "median_house_value": "Valor mediano de la vivienda (USD)"
})
housing_renamed["Máxima similitud con clúster"] = similarities.max(axis=1)
housing_renamed.plot(kind="scatter", x="Longitud", y="Latitud", grid=True,
                     s=housing_renamed["Población"] / 100, label="Población",
                     c="Máxima similitud con clúster",
                     cmap="jet", colorbar=True,
                     legend=True, sharex=False, figsize=(10, 7))
plt.plot(cluster_simil.kmeans_.cluster_centers_[:, 1],
         cluster_simil.kmeans_.cluster_centers_[:, 0],
         linestyle="", color="black", marker="X", markersize=20,
         label="Centros de clúster")
plt.legend(loc="upper right")
plt.show()

## Pipeline de preprocesamiento completo

En este punto, hemos desarrollado todos los componentes necesarios para el pipeline de preprocesamiento completo:

- **Pipelines básicos** ([e2e050](e2e050_pipelines.ipynb)): `SimpleImputer`, `StandardScaler`, `OneHotEncoder` y `ColumnTransformer`
- **Transformadores Personalizados** ([e2e051](e2e051_custom_transformers.ipynb)): `FunctionTransformer` para ratios de características y transformaciones logarítmicas
- **Características geoespaciales** (este notebook): transformador `ClusterSimilarity` usando K-means y el kernel RBF

Estos componentes se combinan en un módulo de preprocesamiento reutilizable en [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py), que es importado por los notebooks posteriores de Entrenamiento del Modelo y evaluación.